(comp/00-algorithms)=
# Coding Problems

![Status](https://img.shields.io/static/v1.svg?label=Status&message=Ongoing&color=orange)
[![Source](https://img.shields.io/static/v1.svg?label=GitHub&message=Source&color=181717&logo=GitHub)](https://github.com/particle1331/ok-transformer/blob/master/docs/nb/comp/00-algorithms.ipynb)
[![Stars](https://img.shields.io/github/stars/particle1331/ok-transformer?style=social)](https://github.com/particle1331/ok-transformer)
[![LeetCode user particle1331](https://img.shields.io/badge/dynamic/json?style=flat&labelColor=black&color=%23ffa116&label=Solved&query=solvedOverTotal&url=https%3A%2F%2Fbadge.xyli.tech/%2Fapi%2Fusers%2Fparticle1331&logo=leetcode&logoColor=yellow)](https://leetcode.com/particle1331/)


---

## Backtracking

Backtracking algorithms are used to systematically explore all possible solutions. The basic idea is to build a solution incrementally, while making choices along the way. At each step, the algorithm explores a possible choice and then recursively explores further choices based on that choice. If a certain choice leads to a dead-end or an invalid solution, the algorithm "backtracks" either by undoing the last choice, or terminating the current solution path, and moving on to the next option. Fast backtracking algorithms are able to quickly determine and thereby terminate incorrect solution paths.

### Reverse Shuffle Merge

**Problem** [[HR](https://www.hackerrank.com/challenges/reverse-shuffle-merge/problem)]. Given a string $S$ containing characters with even counts, we want to find a lexically minimal string $A$ containing half of each characters of $S$ such that $a_i = s_{j_i}$ where $j_{k} \leq j_{k-1} \leq \ldots \leq j_1$ with $k = \frac{1}{2}|S|.$ For example, if $S$ is `eggcegcg` then $A$ is the string `cegg` which can be constructed as `·gg·e·c·` within $S$.

**Solution.** Note that we can iterate over $S$ from right to left choosing to skip or take a character. Since $A$ has to be minimal, we want to take minimal characters, and skip those that are large. However, skipping too often can also result in a suboptimal solution. Consider `S = 'cdeedcaaaa'` here skipping `c` and `d` gets us in trouble since `e` reaches its skip limit of 1/2 its count, and we are forced to take it, getting the suboptimal solution `A = 'aaedc'`. Hence, we want to take characters, unless we are forced to skip, or we have better options:

In [ ]:
from collections import Counter

def solve(S):
    counts = Counter(S)
    chars = counts.keys()
    A = ['.']
    take_count = {c: 0 for c in chars}
    skip_count = {c: 0 for c in chars}

    def take(s):
        A.append(s)
        take_count[s] += 1

    def skip(s):
        skip_count[s] += 1

    def squash(s):
        while (len(A) > 1) and (A[-1] > s) and (skip_count[A[-1]] < counts[A[-1]] / 2):
            skip_count[A[-1]] += 1
            take_count[A[-1]] -= 1
            A.pop()
        take(s)

    for s in S[::-1]:        
        if take_count[s] == counts[s] / 2:
            skip(s)
        else:
            squash(s)

    return "".join(A[1:])


S = "eggcegcg"
A = solve(S)
A

Notice that instead of `take(s)` we have `squash(s)`. The trick is to construct chains: `a <= a <= c <= d` and  if we encounter `b` such that `d > b`, then we backtrack by popping `d` and `c` assuming the skip limit allows it. Our sequence then becomes `a <= a <= b`. The equals is important here to allow consecutive same characters. This is also why we added `.` at the start which is smaller than any character. 

Observe that the `squash` function only takes into account `A` and ignores skipped characters. This is fine since instances of these characters are already optimally placed in the chain. It only adds more skipped characters which are suboptimal in the chain compared to the current queried character. Trying a more complicated input:

In [ ]:
def viz(S):
    chars = Counter(S)
    d = {c: [] for c in sorted(chars.keys(), reverse=True)}

    for s in S[::-1]:
        for c in d.keys():
            if c == s:
                d[c].append(s + ' ')
            else:
                d[c].append('  ')

    for c in d.keys():
        print(''.join(d[c]))



S = "dcadeebcccaaba."
viz(S)
print()
print(f"S = {S[:-1]}")
print("A =", solve(S)[1:])

```{figure} ../../img/comp/greedy-1-soln-1.png
---
width: 60%
---
To respect optimality, we try as much as possible to have our curve in the bottom half of the diagonal. At the start, `b` is squashed after getting `a <= b ? a` while the third `a` skipped. Then, we get consecutive `c <= c ? b` (last `c` is skipped due to take limit) only the latter `c` is popped due to skip limit. Next, we encounter two `e`, the latter skipped due to take limit. And so on. Marked characters are skipped while unmarked characters were squashed (essentially also skipped).
```

### Generate Parenthesis

**Problem** [[LC#22](https://leetcode.com/problems/generate-parentheses/description/?envType=study-plan-v2&envId=top-100-liked)]. Given $n$ pairs of parentheses, generate all strings of well-formed parentheses. For $n = 1$, the only solution is `'()'`.

**Solution.** Suppose you have a string of well-formed parentheses, then you can only add `(`. If you have `(` in excess, you can add enough `)` or add more `(` until its count reaches $n$. Hence, we can just branch off at each candidate solution by adding `(` or `)` while keeping constraints in check. This ultimately results in well-formed parentheses at the leaves. 

In [ ]:
def solve(n):
    out = [(0, 0, "")]
    while True:
        tmp = []
        for res in out:
            l, r, s = res
            if l < n:
                tmp.append((l + 1, r, s + "("))
            if l > r:
                tmp.append((l, r + 1, s + ")"))
        
        if len(tmp) == 0:
            return [s for (l, r, s) in out]
        else:
            out = tmp


print(solve(1))
print(solve(3))

**Remark.** Candidate solutions are represented as tuples `(l, r, s)` containing left parenthesis count `l`, right parenthesis count `r`, and the constucted string `s`.

<br>

```{figure} ../../img/comp/parenthesis-perf.png
---
width: 80%
---
```

### Combination Sum

**Problem** [[LC#39](https://leetcode.com/problems/combination-sum/?envType=study-plan-v2&envId=top-100-liked)]. Given a sequence $S = (s_1, \ldots, s_n)$ of distinct integers and a target $t$, generate all sequences $(s_{j_1}, \ldots, s_{j_k})$ such that $s_{j_1} \leq \ldots \leq s_{j_k}$ and $s_{j_1} + \ldots + s_{j_k} = t$. For example, let $S = (2, 3, 5)$ and $t = 8.$ Then, the expected solution is the set of sequences $(2, 2, 2, 2)$, $(2, 3, 3)$, and $(3, 5).$

**Solution.** Our solution iteratively constructs paths (sequences starting from single integers) from previous paths which are still open, i.e. those whose sum does not exceed or equal the target. This would make more sense by looking at the algorithm for the given example:

```{margin}
- `?` = open
- `x` = exceeds target
- `/` = equals target
```
```
depth = 1
5 ?
3 ?
2 ?

depth = 2
5 5 x
5 3 /
5 2 ?
3 3 ?
3 2 ?
2 2 ?

depth = 3
5 2 2 x
3 3 3 x
3 3 2 x
3 2 2 ?
2 2 2 ?

depth = 4
3 2 2 2 x
2 2 2 2 /
```

Note that each open path is appended by an integer less than or equal the last integer. This is important for constructing all unique sequences. The algorithm terminates when there are no more open paths. Below we also cache intermediate path values (i.e. its total sum) for faster evaluation (summing two numbers).

In [ ]:
def solve(S, t, reverse=True):
    solution = [[t]] if t in S else []
    S = list(filter(lambda s: s < t, sorted(S, reverse=reverse)))

    paths = list(zip([(i,) for i in range(len(S))], S))

    while len(paths) > 0:
        tmp = []
        for path in paths:
            seq, val = path
            i = seq[-1]
            for j in range(i, len(S)):
                seqj, valj = seq + (j,), val + S[j]
                if valj == t:
                    solution.append([S[k] for k in seqj])
                if valj < t:
                    tmp.append((seqj, valj))
        paths = tmp

    return solution


solve(S=[2, 3, 5, 8], t=8)

**Remark.** It is important to close out larger sequences first. This explains why we reversed `S`. This way we avoid creating long sequences which will exceed the target anyway. For example, we close out `5 2 2 x` early instead of wasting constructing `2 2 2 5 x` if we start with `2` instead of `5`.

In [ ]:
%%time
S = [24, 16, 30, 7, 5, 4, 9, 29, 8, 35, 3, 17]
t = 29
solve(S, t);

In [ ]:
%%time
solve(S, t, reverse=False);

<br>

```{figure} ../../img/comp/combination-sum.png
---
width: 70%
---
```

### Permutations

**Problem** [[LC#46](https://leetcode.com/problems/permutations/)]. Generate all permutations of a sequence of distinct integers $S.$ 

**Solution.** Our solution is construct sequences starting from singletons and appending these with integers that in $S$ that are not yet in the sequence. Each intermediate sequence $(s_1, \ldots, s_k)$ where $k \leq n = |S|$ is a node in the tree with root node $(s_1)$ for $s \in S.$ Each node combines the output of its children $(s_1, \ldots, s_k, s_j)$ where $s_j \in S - \{s_1, \ldots, s_k\}$, with the final leaves returning the node when $k = n.$ 

The algorithm is sketched below for a set of three elements `S = [0, 1, 2]`. This generates the tree with root node `[0]`. Note that all possible permutations starting with `0` is covered:

```python
branch([0]) = branch([0] + [1]) + branch([0] + [2])
            = branch([0, 1]) + branch([0, 2])
            = branch([0, 1] + [2]) + branch([0, 2] + [1])
            = branch([0, 1, 2]) + branch([0, 2, 1])
            = [[0, 1, 2]] + [[0, 2, 1]]
            = [[0, 1, 2], [0, 2, 1]]
```

Final solution:

In [ ]:
def solve(S):
    def branch(node):
        if len(node) == len(S):
            return [node]
        return sum([branch(node + [j]) for j in set(S) - set(node)], [])
        
    return sum([branch([s]) for s in S], [])

solve([0, 1, 2])

<br>

```{figure} ../../img/comp/permutation-perf.png
---
width: 80%
---
```

### N-Queens

**Problem** [[LC#51](https://leetcode.com/problems/n-queens/?envType=study-plan-v2&envId=top-100-liked)]. Let $1 \leq n \leq 9.$ Place $n$ queens on an $n \times n$ chessboard such that no two queens attack each other. Return all distinct solutions.

**Solution.** First let us implement a class for the **board state**:

In [ ]:
from pprint import pprint

class NQueens:
    def __init__(self, n):
        self.n = n
        self.queens = set()
        self.blocked = set()

    def place_queen(self, i, j):
        for _ in range(self.n):
            self.blocked.add((i, _))
            self.blocked.add((_, j))

        x, y = i, j
        while (x >= 0) and (y >= 0):
            self.blocked.add((x, y))
            x -= 1
            y -= 1 

        x, y = i, j
        while (x >= 0) and (y < self.n):
            self.blocked.add((x, y))
            x -= 1
            y += 1

        x, y = i, j
        while (x < self.n) and (y >= 0):
            self.blocked.add((x, y))
            x += 1
            y -= 1

        x, y = i, j
        while (x < self.n) and (y < self.n):
            self.blocked.add((x, y))
            x += 1
            y += 1

        self.queens.add((i, j))

    def viz_board(self):
        board = [['□'] * self.n for _ in range(self.n)]
        for i, j in self.blocked:
            board[i][j] = '☒'

        for i, j in self.queens:
            board[i][j] = '♕'

        for row in board:
            print(''.join(row))
        print()

Failed solution since the third row is entirely blocked:

In [ ]:
state = NQueens(n=4)

state.place_queen(0, 0)
state.viz_board()

state.place_queen(1, 2)
state.viz_board()

Looking at the above process already gives us an idea of how to solve this. Note that all valid solutions has exactly one Queen on every row. This allows us to represent a valid solution as an array of length $n.$ In the code below this is `node`. Each Queen placement results in blocked squares in the next row. This limits the next possible placements. Each choice of next placement results in branching separate game states which we pass along recursively. If the next row is completely blocked, the `branch` function returns `[]`.

In [ ]:
from copy import deepcopy

def branch(node, state):
    state = deepcopy(state)
    k = len(node)
    state.place_queen(k-1, node[-1])

    if len(node) == state.n:
        return [node]

    out = []
    for j in range(state.n):
        if not (k, j) in state.blocked:
            out.append(branch(node + [j], state))

    return sum(out, [])


def solve(n):
    state = NQueens(n=n)
    return sum([branch([i], state) for i in range(n)], [])


solution = solve(n=4)
print(solution)

A solution can be visualized by following each Queen placement:

In [ ]:
state = NQueens(n=4)
s = solution[0]
for i, j in enumerate(s):
    state.place_queen(i, j)
    state.viz_board()

```{figure} ../../img/comp/nqueens.png
---
width: 80%
---
Solution is relatively slow and memory inefficient. Probably the board state can be implemented with less bloat. Moreover, notice that solutions have symmetry (e.g. mirror). The tradeoff is that our code is highly readable.
```

### Word Search

**Problem** [[LC#79](https://leetcode.com/problems/word-search/?envType=study-plan-v2&envId=top-100-liked)]. Let $1 \leq m, n \leq 6.$ Given an $m \times n$ of characters and a word $w.$ Determine if $w$ exists in the grid. Meaning $w$ can be constructed using horizontally or vertically adjacent cells. The same letter cell may not be used more than once.

**Solution.** Still using our favorite solution pattern: branching off at nodes given certain conditions. Here node is a list of coordinates. We start our nodes at the coordinates of the starting letter of the given word. Then we branch off at each neighboring node (vertical or horizontal adjacent cells) appending the coordinates to the node. 

If a neighboring cell is not the next letter, the branch is quickly terminated by returning `False`. Otherwise, it continues until the entire word is constructed. The result of branching is a list of Booleans which contains `True` if the word is found. Note that letters already in the node are ruled out (`indices - set(node)`) since we can't reuse cells.

In [ ]:
def solve(board, word):
    m = len(board)
    n = len(board[0])

    def validate(word):
        w = Counter(word)
        b = Counter(''.join(sum(board, [])))
        return all(w[c] <= b[c] for c in w.keys())

    def neighbors(node):
        i, j = node[-1]
        indices = set([
            (min(i+1, m-1), j),
            (max(i-1,   0), j),
            (i, max(j-1,   0)),
            (i, min(j+1, n-1)),
        ])
        return set([(x, y) for x, y in indices - set(node)])

    def decode(node):
        return ''.join([board[x][y] for x, y in node])

    def branch(node):
        if decode(node) == word:
            return [True]

        if decode(node) != word[:len(node)]:
            return [False]
        
        return sum(
            [branch(node + [(x, y)]) for x, y in neighbors(node) - set(node)], 
            [False]
        )

    if not validate(word):
        return False

    init_nodes = [(x, y) for x in range(m) for y in range(n) if board[x][y] == word[0]]
    return any(sum([branch([(x, y)]) for x, y in init_nodes], []))


board = [
    ['A','B','C','E'],
    ['S','F','C','S'],
    ['A','D','E','E']
]
word = 'ABCCED'
solve(board, word)

<br>

```{figure} ../../img/comp/word-search-perf.png
---
width: 80%
---
TLE unless we use `validate`.
```

### Palindrome Partitioning

**Problem** [[LC#131](https://leetcode.com/problems/palindrome-partitioning/)]. Let $s$ be a string such that $1 \leq |s| \leq 16.$ Generate all partitions of $s$ into substrings such that every 
element of the partition is a palindrome. For example, if `s = aab`, then the solution consists of `[a, a, b]` and `[aa, b]`.

**Solution.** Observe that the first partition can be obtained by sliding across the given string: `a`, `aa`, or `aab`. Taking only palindromes, we are left with `a` and `aa`. Then, we repeat this process for the leftover strings `ab` and `b`, respectively. Sliding left to right works since the elements of the partition consists of substrings. Note that this process always returns the partition containing every single character.

In [ ]:
def is_palindrome(s: str):
    return s == s[::-1]

def solve(s, verbose=False):
    def branch(node, s):
        if verbose:
            print(node, s)
        if len(s) == 0:
            return [node]
        return sum(
            [branch(node + [s[:i]], s[i:]) for i in range(1, len(s) + 1) if is_palindrome(s[:i])], 
            []
        )
    return branch([], s)


solve("aab", verbose=True)

**Remark.** Entries of the `node` variable are always valid partitions. The node takes `s[:i]` and passes `s[i:]` as the next string for nonzero `i`, so the resulting partition always sums to the original `s`. Finally, since we scan over all possible next partition, we get all solutions and the constructed solutions are distinct.

<br>

```{figure} ../../img/comp/palindrome-partitioning-perf.png
---
width: 80%
---
```

## Binary Search

The general strategy for binary search is to have pointers for equal sized partitions of a sorted set by using comparison operations. The runtime complexity of a binary search algorithm is generally logarithmic in the sample size.

### Median of Two Sorted Arrays

**Problem** [[LC#4](https://leetcode.com/problems/median-of-two-sorted-arrays)]. Given sorted integer arrays $A$ and $B,$ find the median of the two arrays combined. The solution should run in $O(\log (|A|+|B|))$ time.


**Solution.** Partition $A = A_1 \cup A_2$ such that $A_1 \preceq A_2$ which means $v_1 \leq v_2$ for all $v_1 \in A_1$ and $v_2 \in A_2.$ Do the same with $B = B_1 \cup B_2.$ Our goal is to find a partitioning such that $|A_1| + |B_1| = |A_2| + |B_2| + \delta$ where $\delta = 0$ or $1$ and $A_1, B_1 \preceq A_2$ and $A_1, B_1 \preceq B_2.$ That is, $A_1$ and $B_1$ form the left half of the combined arrays when finding the median. 

It suffices to consider the elements at the point where we divide the arrays. Let $a_1 = \max A_1$, $a_2 = \min A_2$ and $b_1 = \max B_1$, $b_2 = \min B_2.$ Note that these values may be set to $\pm\infty$ when the point of division is at the edges of the arrays. In this case some of the components of the partition may be empty ($\varnothing$). Consider the **invariant** that the left half of the partition of the combined array has a fixed size. This is equal to $|A_1| + |B_1|$ where $A_1$ and $B_1$ are initially taken to be the left halfs when finding the median of $A$ and $B$. This is equal to $\left\lceil {|A|} / {2} \right\rceil$ + $\left\lceil {|B|} / {2} \right\rceil.$  First, an example. Suppose, initially, we have:

```text
step 0
a1 = 0   a2 = 1
b1 = 5   b2 = 6
```

Here, this is not correct, so we have to adjust the partition. Shift the partition in $A$ to the right and that of $B$ to the left:

```text
step 1
a1 = 1   a2 = ?
b1 = ?   b2 = 5
```

In this case, $a_1 \leq b_2$ which is correct. Then, we have to proceed to check if $b_1 > a_2.$ Otherwise, we are done. You can see that the idea is to sort of slide $A$ and $B$ around a fixed separating line, such that we get the expected ordering where everything in the left side of the line is less than or equal to everything on the right side of the line. Visualizing the previous step:

```text
             |                    |
A     [... 0 | 1 ...]   ->  [ 0 1 | ... ]   
B     [... 5 | 6 ...]       [ ... | 5 6 ]   
             |                    |
```

Shifting boundary in $A$ to the right and $B$ to the left by 1 maintains the invariant. Notice that the shift results in having $a_1 \leq b_2$ making this another invariant. Then, we continue the next step by once again checking if $b_1 > a_2.$ 
It remains to show that we can always arrange it so that $a_1 \leq b_2$ initially. Suppose $a_1 > b_2$ and $b_1 > a_2,$ then $a_1 > b_2 \geq b_1 > a_2$ which is a contradiction. So we can always relabel the arrays such that $a_1 \leq b_2$ initially.

Does the algorithm terminate? Recall we always move to the right of $A$ and to the left of $B.$ The algorithm only continues when elements in the left side of $B$ are larger than $a_2.$ If initially $|B_1| \leq |A_2|,$ then the entire $B$ ultimately becomes part of the right partition. The part of $A_2$ that is unexamined is fine since $A_1 \preceq A_2$ while the last $A_1$ forms the entire final left half. And if initially $|B_1| > |A_2|,$ the entire $A$ becomes part of the left partition while the unexamined part of $B_1$ also becomes part of the left partition. This is fine since $B_1 \preceq B_2.$ It follows that the algorithm terminates to the correct state.

Notice that the current algorithm is $O(\min(\frac{1}{2}|B|, \frac{1}{2}|A|)).$ This can be improved by performing a binary search on $B_1.$ Here we have to calculate the largest possible step. The left pointer should always be valid (i.e. not what we are looking for), while the right must always be invalid (which is guaranteed at the start of the interation &mdash; otherwise, we are done):

```text
                      | R       L
A               ... 0 | 1 2 4 6 9
B         1 2 7 8 9 9 | 9 ...
            L       R |
            𐄂       ✔
```

Checking the midpoint:
```text
step 1:
                      | R   ?   L
A               ... 0 | 1 2 4 6 9
B         1 2 7 8 9 9 | 9 ...
            L   ?   R |
            𐄂   ✔   ✔
```

Here `8 > 4` so we move the right pointer there. Otherwise, we move the left pointer there.
To recap, initially set `L` to be the largest
step possible and `R` to be beside the cutpoint. 
If the pair in `L` is invalid, we are done since this is the final cutpoint, 
shifting $A$ and $B$ accordingly. Otherwise, we check the midpoint. If valid, we move `L` there
since ${b_1}^\prime \leq b_1 \leq a_2 \leq {a_2}^\prime$,
else we move `R` there since this is a better cutpoint. 
Each step divides the search space into two, resulting in a solution that is in 
$O(\log(\min(|A|, |B|)))$ time.

**Implementation.** Calculating the median once the partitions are determined is trivial. 
The problem assumes that $|A|, |B| \geq 0$ and $|A| + |B| \geq 1.$ Since indexes are annoying if $\min(|A|, |B|) < 10$ we binary insert the shorter array into the longer array using `bisect.insort`, then use `np.median` to compute the median in logarithmic time.

In [ ]:
import math
import bisect
import numpy as np

A = [1, 3]
B = [2, 4]


def solve(A, B):
    if min(len(A), len(B)) < 10:
        if len(A) < len(B):
            for a in A:
                bisect.insort(B, a)
            return np.median(B)
        else:
            for b in B:
                bisect.insort(A, b)
            return np.median(A)

    
    

solve(A, B)

In [ ]:
A = list(range(11))
B = list(range(15))
mA = math.ceil(len(A) / 2)
mB = math.ceil(len(B) / 2)

a1 = A[mA - 1]
a2 = A[mA]
b1 = B[mB - 1]
b2 = B[mB]

if a1 > b2:
    A, B = B, A
    mA, mB = mB, mA
    a1, a2, b1, b2 = b1, b2, a1, a2

s = min(len(A) - mA, len(B) - mB) - 1
rB = mB - 1
lB = mB - 1 - s

def get_indices_a(lB, rB):
    rA = mA + rB - mB + 1
    lA = mA + mB - 1 - lB
    return lA, rA

print(A[:mA], A[mA:])
print(B[:mB], B[mB:])
while lB <= rB:
    rA, lA = get_indices_a(lB, rB)
    print(B[lB], B[rB], A[rA], A[rB])
    lB = (lB + rB) // 2

In [ ]:
print(A[:mA], A[mA:])
print(B[:mB], B[mB:])

In [ ]:
lB = 5
rB = 6
lA, rA = get_indices_a(lB, rB)
lA, rA

In [ ]:
b1 = B[lB]
b2 = B[rB]
a1 = A[lA]
a2 = A[rA]
b1, b2, a1, a2

### 33. Search in Rotated Sorted Arrays

**Problem.** Consider distinct integers
$t_0 < \ldots < t_{n-1} \in \mathbb{Z}.$ Given a rearrangement $S = (t_k, \ldots, t_{n-1}, t_0, \ldots, t_{k-1})$ for $k \geq 1$ and $a \in S,$ find $j$ such that $s_j = a.$ The search algorithm must be $O(\log n).$ 

**Solution.** A solution in $O(n)$ time is to just look at each element of $S$ and compare with $a.$ This actually gets accepted in LC. To get the desired $O(\log n)$ we have to perform some form of binary search.

In [ ]:
import math

def solve(S, a):
    def bisect(j, S):
        n = len(S)
        if n == 1:
            return j if S[0] == a else -1

        if S[0] < S[-1]:
            if a < S[0] or S[-1] < a:
                return -1

        if S[0] > S[-1]:
            if a < S[0] and S[-1] < a:
                return -1

        if a == S[0]:
            return j
        if a == S[-1]:
            return j + (n - 1)

        k = math.ceil(n / 2)
        return max(bisect(j, S[:k]), bisect(j + k, S[k:]))

    return bisect(0, S)


S = [4, 5, 6, 7, 0, 1, 2]
a = 0
solve(S, a)

In [ ]:
import matplotlib.pyplot as plt

S = [8, 9, 10, 11, 1, 2, 3, 4, 6, 7]
n = len(S)
k = math.ceil(n / 2)

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
a = 9
ax[0].axvline(k - 0.5, linestyle='dashed', color='black')
ax[0].scatter(range(len(S)), S)
ax[0].scatter(S.index(a), a, color='red', label='target');

a = 1
ax[1].axvline(k - 0.5, linestyle='dashed', color='black')
ax[1].scatter(range(len(S)), S)
ax[1].scatter(S.index(a), a, color='red', label='target');

a = 6
ax[2].axvline(k - 0.5, linestyle='dashed', color='black')
ax[2].scatter(range(len(S)), S)
ax[2].scatter(S.index(a), a, color='red', label='target');

In [ ]:
S = [6, 7, 8, 9, 10, 11, 1, 2, 3, 4]
n = len(S)
k = math.ceil(n / 2)

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
a = 6
ax[0].axvline(k - 0.5, linestyle='dashed', color='black')
ax[0].scatter(range(len(S)), S)
ax[0].scatter(S.index(a), a, color='red', label='target');

a = 11
ax[1].axvline(k - 0.5, linestyle='dashed', color='black')
ax[1].scatter(range(len(S)), S)
ax[1].scatter(S.index(a), a, color='red', label='target');

a = 3
ax[2].axvline(k - 0.5, linestyle='dashed', color='black')
ax[2].scatter(range(len(S)), S)
ax[2].scatter(S.index(a), a, color='red', label='target');

In [ ]:
S = [7, 8, 9, 10, 11, 1, 2, 3, 4, 6]
n = len(S)
k = math.ceil(n / 2)

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
a = 8
ax[0].axvline(k - 0.5, linestyle='dashed', color='black')
ax[0].scatter(range(len(S)), S)
ax[0].scatter(S.index(a), a, color='red', label='target');

a = 3
ax[1].axvline(k - 0.5, linestyle='dashed', color='black')
ax[1].scatter(range(len(S)), S)
ax[1].scatter(S.index(a), a, color='red', label='target');

In [ ]:
math.ceil(6 / 2 - 1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')

x = np.linspace(-10, 10, 1000)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
y = 1 / (1 + np.exp(-x))
ax[0].plot(x, y, label=r"$\sigma(x) = \frac{1}{1 + e^{-x}}$")
ax[0].grid(linestyle='dotted')
ax[0].set_xlabel('$x$')
ax[0].set_ylabel("$\sigma$")
ax[0].legend();

@np.vectorize
def relu(x):
    return max(0.0, x)

y = relu(x)
ax[1].plot(x, y, label=r"$ReLU(x) = \max(0, x)$")
ax[1].grid(linestyle='dotted')
ax[1].set_xlabel('$x$')
ax[1].set_ylabel("$ReLU$")
ax[1].legend();